# Sessions E1–E5: the EMBED model

Prompts exactly as in the paper. Each prompt cell starts a fresh session with `new_chat()`; a multi-turn conversation continues in the cells below it without `new_chat()`. Where the paper's run answered a question from Dynalyst, the reply is in a cell after the prompt, as `reply('yes')`: it sends the reply only if Dynalyst has asked a question in this run, and otherwise does nothing.

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Run the **setup cell**, choose the model and paste its API key.
2. Run the **files cell**: it copies the extra models, the R01 time course and the papers from `model_files/` (next to this notebook) into `data/`.
3. Run any prompt cell. Prompts that use BioModels, PubMed or OmniPath need internet.
4. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup and files cells again.

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

In [ ]:
# Copy the extra model files, data and papers into the project (only files not already there)
import shutil
FILES = Path("model_files") if Path("model_files").exists() else ROOT / "notebooks" / "benchmarks" / "model_files"
DEST = {".xml": config.DATA_DIR / "sbml_models", ".ode": config.DATA_DIR / "sbml_models",
        ".csv": config.DATA_DIR / "timecourses", ".pdf": config.PAPERS_DIR}
for f in sorted(FILES.iterdir()):
    d = DEST.get(f.suffix.lower())
    if d and not (d / f.name).exists():
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
        print("added", d / f.name)
print("model files ready")

## Examples of Sriram (2026) and Harika and Sriram (2026) (main text: RQ1, Table 2, Figs 9 and 10; S1 File, Section S8)

Each session is a fresh conversation; its prompts are sent in order in the same conversation. P1.1 needs the EMBED paper (Harika and Sriram, ACS Omega 2026) as a PDF in data/papers/; it is not shipped here.

### Session E1: EMBED, symmetric parameter set

In [ ]:
new_chat()
ask('embed2d_set2.ode is the reduced two-variable dual phosphorylation-dephosphorylation model of Harika and Sriram (ACS Omega 2026), with a symmetric parameter set. Draw its bifurcation diagram in ATOT from 0 to 200 for APP and A, and tell me which bifurcations it has and where.')

In [ ]:
ask('Give me the signed interaction network from the Jacobian of embed2d_set2.ode at its steady states for ATOT = 100 and ATOT = 150. Are the signs of the Jacobian entries the same at every steady state, and what feedback loops do they form?')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('Plot the phase plane of embed2d_set2.ode at ATOT = 150 with nullclines and trajectories from several starting points.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

### Session E2: EMBED, bistability plus transcritical set

In [ ]:
new_chat()
ask('Draw the bifurcation diagram of embed2d_set3.ode in ATOT from 0 to 200 and classify every special point it finds.')

In [ ]:
ask('Follow the special points of embed2d_set3.ode in the two parameters ATOT and km5, with km5 from 1 to 60. How do they move?')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
reply('yes')   # Dynalyst asked (widen); reply yes

### Session E3: EMBED, saddle-node set

In [ ]:
new_chat()
ask('Draw the bifurcation diagram of embed2d_set1.ode in ATOT from 0 to 200 and tell me over which ranges of ATOT it is bistable.')

In [ ]:
ask('Check with a steady-state search at ATOT = 80, 95, 110 and 150 how many steady states embed2d_set1.ode has and which are stable.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

### Session E4: EMBED, ensemble scan

In [ ]:
new_chat()
ask('Scan 40 parameter sets of embed2d_set2.ode with km1 and km5 sampled between 51 and 55, continue each in ATOT from 0 to 300, and tell me how many sets show a pitchfork or branch point, a fold, or neither.')

### Session E5: EMBED, full mass-action network

In [ ]:
new_chat()
ask('embed_pdp_massaction.xml is the full mass-action dual phosphorylation-dephosphorylation network behind the EMBED model. Give me its reaction network, conservation laws and deficiency, and whether reaction network theory allows several steady states for some rate constants.')

In [ ]:
ask('Draw its reaction network.')

In [ ]:
ask('Reduce embed_pdp_massaction.xml with the quasi-steady-state approximation for the enzyme complexes and tell me how well the reduced model matches the full one.')